# Case study 3: reported disease associations of gut microbes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaelmdc/braidworks/blob/main/notebooks/case-studies/gut-microbe-disease-associations.ipynb)

**Question.** In how many diseases has each of six well-studied gut species been
reported as differentially abundant, according to the Disbiome database?

**Route.** Species name → NCBI Taxonomy (species taxon ID) → Disbiome (reported
microbe–disease associations). Disbiome is queried from a local database that the
first run builds from the provider's published tables (about 45 MB).

## Setup

Install Braidworks with every public source, then record the environment.

In [ ]:
%pip install -q "braidworks[pandas]" matplotlib

In [ ]:
import datetime
import importlib.metadata
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt

import braidworks as bw

OUT = Path("case-study-output")
OUT.mkdir(exist_ok=True)

# Figure style: one hue, muted axes, hairline grid (print-friendly, colour-blind safe).
INK, MUTED, GRID, ACCENT = "#0b0b0b", "#898781", "#e1e0d9", "#2a78d6"
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False,
})


def record_environment(sources):
    """Versions and date, saved next to the outputs so a run can be reproduced."""
    packages = ["braidworks-core", *sources]
    versions = {name: importlib.metadata.version(name) for name in packages}
    env = {
        "date_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        **versions,
    }
    for key, value in env.items():
        print(f"{key:24} {value}")
    return env


def timed(label, run):
    start = time.perf_counter()
    result = run()
    seconds = time.perf_counter() - start
    print(f"{label}: {seconds:.2f} s  ({result.summary()})")
    return result, seconds

In [ ]:
env = record_environment(['ncbi_weaver', 'disbiome_weaver'])

## Build the local Disbiome database

This downloads Disbiome's tables once and reuses them on later runs. Its time is
reported separately from the lookups.

In [ ]:
from braidworks.core import BraidRegistry
from disbiome_weaver import build_disbiome_weaver_configured
from ncbi_weaver.factory import build_ncbi_weaver

start = time.perf_counter()
disbiome = build_disbiome_weaver_configured(
    db_path=str(OUT / "disbiome.sqlite"), auto_setup=True
)
print(f"Disbiome database ready in {time.perf_counter() - start:.1f} s")


def registry():
    sources = BraidRegistry()
    sources.register(build_ncbi_weaver())
    sources.register(disbiome)
    return sources

## The request

In [ ]:
SPECIES = [
    "Faecalibacterium prausnitzii", "Akkermansia muciniphila", "Escherichia coli",
    "Bacteroides fragilis", "Fusobacterium nucleatum", "Roseburia intestinalis",
]
WANT = ["ncbi.taxon.id", "microbe.disease.count", "microbe.disease.names"]

## Run it, cold and warm

In [ ]:
cache = OUT / "gut-microbe-disease-associations-cache.sqlite"
cache.unlink(missing_ok=True)
session = bw.Session(registry=registry(), cache=cache)
result, cold_s = timed("cold", lambda: session.get(SPECIES, have="organism.name", want=WANT))
_, warm_s = timed("warm", lambda: session.get(SPECIES, have="organism.name", want=WANT))
for braid in result.route:
    print(braid)

In [ ]:
table = result.to_pandas()[["input", "status", "ncbi.taxon.id", "microbe.disease.count"]]
table = table.rename(columns={"input": "species", "microbe.disease.count": "diseases"})
table.to_csv(OUT / "gut-microbe-disease-associations.csv", index=False)
result.to_csv(OUT / "gut-microbe-disease-associations-full.csv")
table

A count is the number of distinct diseases with at least one reported experiment, in
either direction (more or less abundant). It measures how much a species has been
studied as much as how often it is associated with disease.

## Figure

In [ ]:
ordered = table.sort_values("diseases")
fig, ax = plt.subplots(figsize=(5.2, 2.8))
bars = ax.barh(ordered["species"], ordered["diseases"], height=0.55, color=ACCENT)
ax.bar_label(bars, padding=3, fontsize=7, color=INK)
ax.set_xlabel("Diseases with a reported association (Disbiome)")
ax.tick_params(axis="y", length=0)
for label in ax.get_yticklabels():
    label.set_style("italic")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(OUT / "gut-microbe-disease-associations.png")
fig.savefig(OUT / "gut-microbe-disease-associations.svg")

In [ ]:
import json

env["timings_s"] = {"cold": round(cold_s, 2), "warm": round(warm_s, 2)}
(OUT / "gut-microbe-disease-associations-environment.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env["timings_s"]))

## Sources to cite

In [ ]:
print(result.cite())
session.close()

## Reproducing this case study

The sources are live databases, so counts change as they are updated: compare your
numbers with the environment record and date saved in `gut-microbe-disease-associations-environment.json`, not
with the figures in a paper. Timings depend on the network and on provider load; the
warm run shows what a lookup cache saves when an analysis is rerun.